In [1]:
!pip install langchain-experimental langchain-openai pandas requests==2.32.4
!pip install gdown

INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ---------------------------------------- 0.0/2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph 1.2.12 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph-sdk 0.4.5 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip insta


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os
import sys
import gdown
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

# ==========================================
#  PART 1: AUTOMATIC FILE DOWNLOADER
# ==========================================

files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

print("--- Downloading Files from Google Drive ---")
for filename, url in files_to_download.items():
    if not os.path.exists(filename):
        gdown.download(url, filename, quiet=False)
        print(f"Downloaded: {filename}")
    else:
        print(f"Skipped: {filename} (Already exists)")
print("--- Download Complete ---\n")


# ==========================================
#  PART 2: AI AGENT SETUP (MULTI-FILE)
# ==========================================

# 1. SETUP: Get API Key Securely
#print("ENTER YOUR OPENAI API KEY BELOW:")
#api_key = getpass()

print("Initializing OpenAI client...")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("ERROR: OPENAI_API_KEY is not set.")
    sys.exit(1)

print("OpenAI API key found.")

# 2. LOAD ALL CSVs INTO A LIST
dataframes = [] # We will store all the loaded tables here
loaded_names = []

try:
    for filename in files_to_download.keys():
        df = pd.read_csv(filename)
        dataframes.append(df)
        loaded_names.append(filename)
        print(f"SUCCESS: Loaded '{filename}' ({len(df)} rows)")

except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()

custom_prefix = """
You are a master data analyst working with 4 specific DataFrames.
Your goal is to query the correct dataframe to answer user questions.

Here is your data map:
- df1: SaaS Documentation (Technical docs, API usage)
- df2: Credit Card Terms (Financial terms, interest rates, fees)
- df3: Hospital Policy (Visiting rules, admin info)
- df4: Ecommerce FAQs (Shipping, returns, product info)

RULES:
1. First, reason about which DataFrame is most relevant to the question.
2. Use Python string methods like `.str.contains('keyword', case=False)` to find relevant rows.
3. DO NOT look for exact matches; always use flexible string matching.
4. If you find the answer, print it clearly.
"""

# 3. DEFINE THE RULES
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

# 4. INITIALIZE THE AGENT WITH MULTIPLE DFS
try:
    llm = ChatOpenAI(
        temperature=0.0,
        model="gpt-4o-mini",
        openai_api_key=api_key
    )

    # --- KEY CHANGE: We pass the LIST 'dataframes' instead of a single 'df' ---
    agent = create_pandas_dataframe_agent(
       llm,
        dataframes,
        verbose=True,
        agent_type="openai-tools", # 'openai-tools' is often more stable than 'functions'
        prefix=custom_prefix,      # <--- Injecting the map here
        include_df_in_prompt=True, # Ensures the agent sees column names
        #handle_parsing_errors=True, # <--- Allows agent to self-correct coding errors
        allow_dangerous_code=True
    )

    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()

--- Downloading Files from Google Drive ---
Skipped: saas_docs.csv (Already exists)
Skipped: credit_card_terms.csv (Already exists)
Skipped: hospital_policy.csv (Already exists)
Skipped: ecommerce_faqs.csv (Already exists)
--- Download Complete ---

Initializing OpenAI client...
OpenAI API key found.
SUCCESS: Loaded 'saas_docs.csv' (15 rows)
SUCCESS: Loaded 'credit_card_terms.csv' (15 rows)
SUCCESS: Loaded 'hospital_policy.csv' (15 rows)
SUCCESS: Loaded 'ecommerce_faqs.csv' (15 rows)

AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


In [3]:
# ==========================================
#  PART 3: CHAT LOOP
# ==========================================
print("\nType 'exit' or 'quit' to stop conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    final_query = system_prompt + "\n\nQuestion: " + user_input
    print("AI is thinking...")

    try:
        response = agent.invoke(final_query)['output']
        print(f"AI: {response}\n" + "-"*30)
    except Exception as e:
        print(f"An error occurred: {e}")


Type 'exit' or 'quit' to stop conversation.

AI is thinking...


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df4[df4['Question'].str.contains('return policy', case=False) & df4['Question'].str.contains('electronics', case=False)]"}`


    ID Category                                    Question  \
1  102  Returns  What is the return policy for electronics?   

                                              Answer  \
1  Electronics can be returned within 30 days of ...   

                Related Products Last Updated  
1  Laptops, Smartphones, Tablets   2023-09-15  The return policy for electronics states that they can be returned within 30 days of receipt. The item must be in its original packaging with all accessories. If the seal is broken, a 15% restocking fee applies.

> Finished chain.
AI: The return policy for electronics states that they can be returned within 30 days of receipt. The item must be in its original packaging with all accessor